# 🎮 Player Churn Analysis and Prediction

In this notebook, we explore the `online_gaming_behavior_dataset.csv`. 
Our goal is to understand what factors lead to a player being 'highly engaged' versus 'churning' (low engagement), and to build a predictive model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('online_gaming_behavior_dataset.csv')
df.head()

### Data Cleaning and Preprocessing
We will drop the `PlayerID` and map the target variable `EngagementLevel` to a binary Churn indicator. Low engagement means the player is churning.

In [2]:
df = df.drop(columns=['PlayerID'])
df['Churn'] = df['EngagementLevel'].apply(lambda x: 1 if x == 'Low' else 0)
df = df.drop(columns=['EngagementLevel'])

### Encoding Categorical Variables
We convert categorical features like Gender, Location, and Genre to numeric.

In [3]:
gender_map = {val: idx for idx, val in enumerate(df['Gender'].unique())}
df['Gender'] = df['Gender'].map(gender_map)

location_map = {val: idx for idx, val in enumerate(df['Location'].unique())}
df['Location'] = df['Location'].map(location_map)

genre_map = {val: idx for idx, val in enumerate(df['GameGenre'].unique())}
df['GameGenre'] = df['GameGenre'].map(genre_map)

diff_map = {val: idx for idx, val in enumerate(df['GameDifficulty'].unique())}
df['GameDifficulty'] = df['GameDifficulty'].map(diff_map)

### Model Training
Let's split the dataset, scale our features, and train a Random Forest Classifier.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

### Feature Importance
What features drive player churn the most?

In [5]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title('Feature Importances for Player Churn')
plt.show()

### Save Model and Scaler

In [6]:
import joblib
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Models saved successfully.')